# Đồ Án Machine Learning: Phân Loại Biển Báo Giao Thông
## Notebook 03: Xây Dựng & Huấn Luyện Mô Hình Cơ Sở (Baseline Model - Custom CNN)

---
### Mục tiêu thực nghiệm:
1. **Thiết lập mô hình cơ sở (Baseline Model):** Tự thiết kế một mạng tích chập sâu (Deep Convolutional Neural Network - `TrafficSignCNN`) từ đầu bằng PyTorch để làm mốc đánh giá chuẩn (benchmark) cho bài toán 250 lớp biển báo.
2. **Phân tích kiến trúc mạng:**
   - 3 khối tích chập lồng ghép (*Convolutional Blocks*): `Conv2D` $ightarrow$ `BatchNorm2d` $ightarrow$ `ReLU` $ightarrow$ `MaxPool2d` $ightarrow$ `Dropout`.
   - Khối phân loại (*Fully Connected Classifier*): `AdaptiveAvgPool2d` $ightarrow$ `Linear` $ightarrow$ `BatchNorm1d` $ightarrow$ `Dropout` $ightarrow$ `Linear(num_classes=250)`.
3. **Huấn luyện mô hình:**
   - Sử dụng hàm mất mát `CrossEntropyLoss` và thuật toán tối ưu `Adam` kết hợp `ReduceLROnPlateau`.
   - Đánh giá sự hội tụ thông qua đồ thị Loss và Accuracy qua các Epoch.
4. **Đánh giá sơ bộ:** Kiểm thử độ chính xác trên tập dữ liệu Test độc lập và phân tích ưu/nhược điểm của mô hình tự thiết kế.

In [ ]:
import sys
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

project_root = Path('..').resolve()
if not (project_root / 'data').exists():
    project_root = Path('.').resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import load_config
from src.data.dataset import TrafficSignDataset
from src.data.transforms import get_train_transforms, get_val_transforms
from src.models.custom_cnn import TrafficSignCNN
from src.training.train import train_one_epoch, validate
from src.utils.metrics import calculate_classification_metrics

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[+] Thiết bị tính toán đang sử dụng: {device}")

### 1. Phân Tích Kiến Trúc Mạng `TrafficSignCNN`
Mạng CNN được thiết kế theo nguyên lý kim tự tháp đặc trưng:
- Tăng dần số lượng bộ lọc (Channels: $32 ightarrow 64 ightarrow 128$) để nắm bắt từ các đặc trưng biên cạnh cơ bản đến hình học biển báo phức tạp.
- Sử dụng `BatchNorm` sau mỗi lớp Conv để giảm hiện tượng trôi dạt hiệp biến nội (Internal Covariate Shift).
- Đặt `Dropout` có trọng số tăng dần ($0.15 ightarrow 0.3$) nhằm chống hiện tượng Overfitting khi số lượng lớp lên tới 250.

In [ ]:
num_classes = 250
model = TrafficSignCNN(num_classes=num_classes, dropout_rate=0.3).to(device)

# Đếm số lượng tham số học được (Trainable Parameters)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
conv_params = sum(p.numel() for p in model.features.parameters() if p.requires_grad)
fc_params = sum(p.numel() for p in model.classifier.parameters() if p.requires_grad)

print(f"============================================================")
print(f"  KIẾN TRÚC MÔ HÌNH BASELINE: TrafficSignCNN")
print(f"============================================================")
print(model)
print(f"------------------------------------------------------------")
print(f"Tổng số tham số học được:    {total_params:,} tham số (~{total_params*4 / (1024*1024):.2f} MB)")
print(f" - Tham số khối Tích chập (Features):   {conv_params:,} ({conv_params/total_params*100:.1f}%)")
print(f" - Tham số khối Phân loại (Classifier): {fc_params:,} ({fc_params/total_params*100:.1f}%)")
print(f"============================================================")

### 2. Chuẩn Bị Dữ Liệu Huấn Luyện (Data Pipeline)
Nạp dữ liệu đã xử lý từ `data/processed/dataset_summary.csv` và khởi tạo DataLoaders.

In [ ]:
data_dir = project_root / 'data'
summary_csv = data_dir / 'processed' / 'dataset_summary.csv'

train_transform = get_train_transforms(img_size=(64, 64))
val_transform = get_val_transforms(img_size=(64, 64))

train_dataset = TrafficSignDataset(summary_csv, base_dir=data_dir, split='train', transform=train_transform)
val_dataset = TrafficSignDataset(summary_csv, base_dir=data_dir, split='val', transform=val_transform)
test_dataset = TrafficSignDataset(summary_csv, base_dir=data_dir, split='test', transform=val_transform)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"[+] Dữ liệu sẵn sàng:")
print(f"    - Tập Huấn luyện (Train): {len(train_dataset)} mẫu ({len(train_loader)} batches)")
print(f"    - Tập Xác thực (Val):     {len(val_dataset)} mẫu ({len(val_loader)} batches)")
print(f"    - Tập Kiểm thử (Test):    {len(test_dataset)} mẫu ({len(test_loader)} batches)")

### 3. Cấu Hình Huấn Luyện (Loss, Optimizer & Scheduler)

In [ ]:
criterion = nn.CrossEntropyLoss()
learning_rate = 0.001
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

print(f"[+] Loss Function: CrossEntropyLoss")
print(f"[+] Optimizer: Adam (lr={learning_rate}, weight_decay=1e-4)")
print(f"[+] Scheduler: ReduceLROnPlateau (factor=0.5, patience=2)")

### 4. Huấn Luyện Mô Hình Hoặc Nạp Checkpoint Đã Tối Ưu
Chúng ta sẽ tiến hành huấn luyện mô hình qua các epoch (hoặc nạp trọng số tối ưu từ `checkpoints/best_model.pth` nếu đã huấn luyện xong từ trước).

In [ ]:
checkpoint_path = project_root / 'checkpoints' / 'best_model.pth'
epochs = 3
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

# Kiểm tra xem đã có checkpoint lưu sẵn chưa
if checkpoint_path.exists():
    state = torch.load(checkpoint_path, map_location=device, weights_only=False)
    m_name = state.get('model_name') if isinstance(state, dict) else None
    if m_name == 'custom_cnn':
        model.load_state_dict(state['model_state_dict'])
        print(f"[+] Đã khôi phục trọng số Custom CNN từ checkpoint (Accuracy: {state.get('best_acc', 0.0)*100:.2f}%)!")
    else:
        print(f"[i] Checkpoint hiện có là của kiến trúc '{m_name}', mô hình Baseline Custom CNN sẽ được huấn luyện mới.")

print(f"\n[+] Bắt đầu quá trình đánh giá/huấn luyện Baseline Model ({epochs} epochs)...")
start_time = time.time()

for epoch in range(1, epochs + 1):
    t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    v_loss, v_acc = validate(model, val_loader, criterion, device)
    scheduler.step(v_loss)
    
    history['train_loss'].append(t_loss)
    history['train_acc'].append(t_acc)
    history['val_loss'].append(v_loss)
    history['val_acc'].append(v_acc)
    
    print(f"Epoch [{epoch:02d}/{epochs:02d}] - Train Loss: {t_loss:.4f} | Train Acc: {t_acc*100:.2f}% | Val Loss: {v_loss:.4f} | Val Acc: {v_acc*100:.2f}%")

total_time = time.time() - start_time
print(f"[+] Hoàn thành sau: {total_time:.1f} giây!")

### 5. Trực Quan Hóa Quá Trình Học (Learning Curves)
Đồ thị Loss và Accuracy cho thấy khả năng hội tụ và tốc độ học của mô hình Custom CNN.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history['train_loss']) + 1)

# Đồ thị hàm mất mát Loss
axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'], 'r--s', label='Val Loss', linewidth=2)
axes[0].set_title('Đồ Thị Hàm Mất Mát (Loss Curve)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Cross-Entropy Loss', fontsize=11)
axes[0].legend()
axes[0].grid(True)

# Đồ thị độ chính xác Accuracy
axes[1].plot(epochs_range, [acc*100 for acc in history['train_acc']], 'b-o', label='Train Accuracy', linewidth=2)
axes[1].plot(epochs_range, [acc*100 for acc in history['val_acc']], 'g--s', label='Val Accuracy', linewidth=2)
axes[1].set_title('Đồ Thị Độ Chính Xác (Accuracy Curve)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Độ chính xác (%)', fontsize=11)
axes[1].legend()
axes[1].grid(True)

plt.suptitle("Đánh Giá Quá Trình Huấn Luyện Mô Hình Baseline CNN", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 6. Đánh Giá Trực Quan Trên Tập Kiểm Thử (Test Set)
Hiển thị ngẫu nhiên 8 mẫu trên tập Test cùng kết quả dự đoán của mô hình cơ sở.

In [ ]:
model.eval()
test_loss, test_acc = validate(model, test_loader, criterion, device)
print(f"[+] Độ chính xác trên toàn bộ tập Test: {test_acc*100:.2f}% | Test Loss: {test_loss:.4f}")

df_labels = pd.read_csv(data_dir / 'class_labels.csv')
images_sample, labels_sample = next(iter(test_loader))
images_sample, labels_sample = images_sample.to(device), labels_sample.to(device)

with torch.no_grad():
    outputs = model(images_sample)
    _, preds = outputs.max(1)

def unnormalize(tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = tensor.cpu().permute(1, 2, 0).numpy()
    return np.clip(img * std + mean, 0, 1)

fig, axes = plt.subplots(1, 8, figsize=(18, 3.5))
for i in range(8):
    true_id = labels_sample[i].item()
    pred_id = preds[i].item()
    is_correct = (true_id == pred_id)
    
    true_code = df_labels[df_labels['class_id'] == true_id].iloc[0]['sign_code']
    pred_code = df_labels[df_labels['class_id'] == pred_id].iloc[0]['sign_code']
    
    axes[i].imshow(unnormalize(images_sample[i]))
    title_color = 'green' if is_correct else 'red'
    axes[i].set_title(f"True: {true_code}\nPred: {pred_code}", fontsize=9, color=title_color, fontweight='bold')
    axes[i].axis('off')

plt.suptitle("Kiểm Thử Ngẫu Nhiên Trên 8 Mẫu Test (Xanh: Đúng, Đỏ: Sai)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 7. Nhận Xét & Kết Luận
* **Ưu điểm:**
  - Mô hình `TrafficSignCNN` có kiến trúc gọn nhẹ (~1.3 triệu tham số), tốc độ xử lý nhanh, thích hợp chạy thử nghiệm nhanh.
  - Hội tụ ổn định nhờ các lớp `BatchNorm2d` và kỹ thuật điều hòa `Dropout`.
* **Hạn chế:**
  - Do học đặc trưng từ đầu (*from scratch*) trên tập dữ liệu gồm tới 250 lớp, mô hình baseline gặp thách thức ở các biển báo có độ tương đồng cao hoặc biển báo bị che khuất một phần.
  - Đây là cơ sở thực nghiệm để chúng ta bước sang **Notebook 04**, ứng dụng các mô hình học chuyển giao **Transfer Learning (MobileNetV3 / ResNet18)** nhằm nâng cao độ chính xác và khả năng tổng quát hóa.